# Sesión 8 — Machine Learning tradicional con MLflow

## Del dashboard al laboratorio de Machine Learning

En la Sesión 7 publicamos la capa Gold, KPIs y dashboards. En esta sesión convertimos ese producto analítico en un laboratorio reproducible de modelos con MLflow.

**Pregunta guía:** ¿podemos estimar el bagazo entregado por ingenio usando lluvia, caña molida, estacionalidad e histórico reciente?

**Caso principal:** Bagazo.  
**Caso complementario:** Lumi, solo como reto opcional de experiencia/delivery.

> MLflow convierte el entrenamiento de modelos en un proceso trazable, comparable, auditable y gobernable.


## 0. Reglas de seguridad

- No modificar `workspace.bagazo_gold.*`.
- No modificar `workspace.lumi_gold.*`.
- No usar `workspace.delta_lab.*` como fuente de ML.
- Crear únicamente schemas y tablas nuevas bajo `workspace.ml_*`.


In [0]:
# ==========================================================
# Configuración general de la Sesión 8
# ==========================================================
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *

import pandas as pd
import numpy as np
import os
import json
import tempfile
import warnings
warnings.filterwarnings("ignore")

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance

import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
try:
    from mlflow.models.signature import infer_signature
except Exception:
    infer_signature = None

CATALOG = "workspace"
SOURCE_BAGAZO = f"{CATALOG}.bagazo_gold.fact_operacion_ingenios"
CONTROL_GOLD = f"{CATALOG}.control.gold_publication_summary_sesion_07"

SCHEMA_FEATURES = f"{CATALOG}.ml_features"
SCHEMA_MODELS = f"{CATALOG}.ml_models"
SCHEMA_MONITORING = f"{CATALOG}.ml_monitoring"

TABLE_FEATURES = f"{SCHEMA_FEATURES}.bagazo_features_training"
TABLE_SUMMARY = f"{SCHEMA_MODELS}.bagazo_experiment_summary_sesion_08"
TABLE_HOLDOUT = f"{SCHEMA_MONITORING}.bagazo_holdout_predictions_sesion_08"
TABLE_REGISTRY_LOG = f"{SCHEMA_MODELS}.model_registry_attempt_log_sesion_08"

RANDOM_STATE = 42
TEST_SIZE = 0.20
ENABLE_OPTIONAL_GRADIENT_BOOSTING = False
ENABLE_UC_MODEL_REGISTRY = False

print("✅ Configuración cargada")
print(f"Fuente principal: {SOURCE_BAGAZO}")
print(f"Tabla de features: {TABLE_FEATURES}")


## 1. Validar fuentes Gold

El modelo parte de Gold, no de Bronze. Esta es una decisión de trazabilidad y gobierno.


In [0]:
# ==========================================================
# 1. Validar fuentes Gold y control
# ==========================================================
def table_exists(full_name: str) -> bool:
    try:
        return spark.catalog.tableExists(full_name)
    except Exception:
        try:
            spark.sql(f"DESCRIBE TABLE {full_name}")
            return True
        except Exception:
            return False

required_tables = [SOURCE_BAGAZO]
for tbl in required_tables:
    if not table_exists(tbl):
        raise ValueError(f"❌ No existe la tabla requerida: {tbl}. Ejecuta primero la Sesión 7.")
    print(f"✅ Existe: {tbl}")

bagazo_raw = spark.table(SOURCE_BAGAZO)
print("Columnas disponibles en Gold Bagazo:")
print(bagazo_raw.columns)

conteo = bagazo_raw.agg(
    F.count("*").alias("filas")
)
display(conteo)

# Validación de granularidad fecha + ingenio. Se resuelven nombres de columna de forma robusta.
def resolve_col(df, candidates, required=True):
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    if required:
        raise ValueError(f"No encontré ninguna columna candidata: {candidates}. Columnas actuales: {df.columns}")
    return None

COL_FECHA = resolve_col(bagazo_raw, ["fecha", "date", "fecha_operacion"])
COL_INGENIO = resolve_col(bagazo_raw, ["ingenio", "planta", "nombre_ingenio"])
COL_LLUVIA = resolve_col(bagazo_raw, ["lluvia_mm", "promedio_lluvias_mm", "promedio_lluvia_mm", "lluvia", "precipitacion_mm"])
COL_CANA = resolve_col(bagazo_raw, ["cana_molida_ton", "caña_molida_ton", "cana_molida_toneladas", "caña_molida_toneladas", "cana_molida"])
COL_BAGAZO = resolve_col(bagazo_raw, ["bagazo_entregado_ton", "bagazo_entregado_toneladas", "bagazo_entregado", "bagazo"])
COL_COMENTARIOS = resolve_col(bagazo_raw, ["comentarios", "comentario", "comentarios_operativos", "comentarios_operacion"], required=False)
COL_RIESGO = resolve_col(bagazo_raw, ["riesgo_bajo_bagazo", "target_riesgo_bajo_bagazo", "flag_riesgo_bajo_bagazo"], required=False)

print("\nColumnas resueltas:")
for k, v in {
    "fecha": COL_FECHA,
    "ingenio": COL_INGENIO,
    "lluvia": COL_LLUVIA,
    "caña": COL_CANA,
    "bagazo": COL_BAGAZO,
    "comentarios": COL_COMENTARIOS,
    "riesgo": COL_RIESGO
}.items():
    print(f"- {k}: {v}")

granularidad = bagazo_raw.select(
    F.col(COL_FECHA).alias("fecha"),
    F.col(COL_INGENIO).alias("ingenio")
).agg(
    F.count("*").alias("filas"),
    F.countDistinct("fecha", "ingenio").alias("combinaciones_fecha_ingenio")
)
display(granularidad)

if table_exists(CONTROL_GOLD):
    print("✅ Tabla de control Gold disponible")
    display(spark.table(CONTROL_GOLD).limit(20))
else:
    print("⚠️ No se encontró la tabla de control Gold. El flujo puede continuar, pero revisa la Sesión 7.")


In [0]:
# ==========================================================
# 2. Crear schemas ML sin tocar Gold
# ==========================================================
for schema_name in [SCHEMA_FEATURES, SCHEMA_MODELS, SCHEMA_MONITORING]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
    print(f"✅ Schema listo: {schema_name}")


## 2. Feature engineering Bagazo

Construiremos variables temporales, operativas e históricas por `ingenio`. Las ventanas móviles usan únicamente días anteriores.


In [0]:
# ==========================================================
# 3. Feature engineering Bagazo con PySpark
# ==========================================================
comentarios_expr = F.lit("") if COL_COMENTARIOS is None else F.coalesce(F.col(COL_COMENTARIOS).cast("string"), F.lit(""))

base = (
    bagazo_raw
    .select(
        F.to_date(F.col(COL_FECHA)).alias("fecha"),
        F.col(COL_INGENIO).cast("string").alias("ingenio"),
        F.col(COL_LLUVIA).cast("double").alias("lluvia_mm"),
        F.col(COL_CANA).cast("double").alias("cana_molida_ton"),
        F.col(COL_BAGAZO).cast("double").alias("bagazo_entregado_ton"),
        comentarios_expr.alias("comentarios_operativos")
    )
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("ingenio").isNotNull())
)

# Umbral de riesgo bajo por ingenio: si no viene desde Gold, se estima con percentil 25 por ingenio.
if COL_RIESGO is not None:
    riesgo_df = bagazo_raw.select(
        F.to_date(F.col(COL_FECHA)).alias("fecha"),
        F.col(COL_INGENIO).cast("string").alias("ingenio"),
        F.col(COL_RIESGO).cast("int").alias("target_riesgo_bajo_bagazo")
    )
    base = base.join(riesgo_df, on=["fecha", "ingenio"], how="left")
else:
    umbrales = base.groupBy("ingenio").agg(
        F.expr("percentile_approx(bagazo_entregado_ton, 0.25)").alias("umbral_riesgo_bajo_bagazo")
    )
    base = (
        base.join(umbrales, on="ingenio", how="left")
        .withColumn(
            "target_riesgo_bajo_bagazo",
            F.when(F.col("bagazo_entregado_ton") <= F.col("umbral_riesgo_bajo_bagazo"), F.lit(1)).otherwise(F.lit(0))
        )
        .drop("umbral_riesgo_bajo_bagazo")
    )

w = Window.partitionBy("ingenio").orderBy("fecha")
w_prev_7 = w.rowsBetween(-7, -1)
w_prev_14 = w.rowsBetween(-14, -1)

features = (
    base
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("dia_semana", F.dayofweek("fecha"))
    .withColumn("dia_mes", F.dayofmonth("fecha"))
    .withColumn("semana_anio", F.weekofyear("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("es_fin_de_semana", F.when(F.col("dia_semana").isin(1, 7), 1).otherwise(0))
    .withColumn("lluvia_alta", F.when(F.col("lluvia_mm") >= 10, 1).otherwise(0))
    .withColumn("temporada_lluviosa", F.when(F.col("mes").isin(4,5,9,10,11), 1).otherwise(0))
    .withColumn("tiene_comentario_operativo", F.when(F.length(F.trim("comentarios_operativos")) > 0, 1).otherwise(0))
    .withColumn("lluvia_lag_1", F.lag("lluvia_mm", 1).over(w))
    .withColumn("lluvia_lag_7", F.lag("lluvia_mm", 7).over(w))
    .withColumn("lluvia_promedio_7d", F.avg("lluvia_mm").over(w_prev_7))
    .withColumn("lluvia_promedio_14d", F.avg("lluvia_mm").over(w_prev_14))
    .withColumn("bagazo_lag_1", F.lag("bagazo_entregado_ton", 1).over(w))
    .withColumn("bagazo_lag_7", F.lag("bagazo_entregado_ton", 7).over(w))
    .withColumn("bagazo_promedio_7d", F.avg("bagazo_entregado_ton").over(w_prev_7))
    .withColumn("cana_lag_1", F.lag("cana_molida_ton", 1).over(w))
    .withColumn("cana_promedio_7d", F.avg("cana_molida_ton").over(w_prev_7))
    .withColumn("target_bagazo_entregado_ton", F.col("bagazo_entregado_ton"))
)

# Filtrar primeras filas sin histórico suficiente para evitar nulos en lags relevantes.
required_feature_cols = [
    "fecha", "ingenio", "anio", "mes", "dia_semana", "dia_mes", "semana_anio",
    "lluvia_mm", "cana_molida_ton", "lluvia_alta", "temporada_lluviosa", "tiene_comentario_operativo",
    "lluvia_lag_1", "lluvia_lag_7", "lluvia_promedio_7d", "lluvia_promedio_14d",
    "bagazo_lag_1", "bagazo_lag_7", "bagazo_promedio_7d", "cana_lag_1", "cana_promedio_7d",
    "target_bagazo_entregado_ton", "target_riesgo_bajo_bagazo"
]

features_training = features.select(*required_feature_cols).dropna(subset=[
    "lluvia_lag_7", "lluvia_promedio_14d", "bagazo_lag_7", "bagazo_promedio_7d", "cana_lag_1", "cana_promedio_7d", "target_bagazo_entregado_ton"
])

features_training.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(TABLE_FEATURES)
print(f"✅ Tabla creada: {TABLE_FEATURES}")

display(spark.table(TABLE_FEATURES).orderBy("ingenio", "fecha").limit(10))


In [0]:
# ==========================================================
# 4. Validaciones del dataset de features
# ==========================================================
features_df = spark.table(TABLE_FEATURES)

resumen_features = features_df.agg(
    F.count("*").alias("filas_features"),
    F.countDistinct("fecha", "ingenio").alias("granularidad_fecha_ingenio"),
    F.countDistinct("ingenio").alias("ingenios"),
    F.min("fecha").alias("fecha_min"),
    F.max("fecha").alias("fecha_max"),
    F.avg("target_bagazo_entregado_ton").alias("target_promedio")
)
display(resumen_features)

nulos = features_df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in features_df.columns
])
display(nulos)


## 3. Escenarios predictivos

Escenario A usa caña del mismo día. Escenario B excluye caña del mismo día para reducir riesgo de leakage operacional.


In [0]:
# ==========================================================
# 5. Escenarios predictivos: Nowcasting vs Forecast sin leakage
# ==========================================================
TARGET = "target_bagazo_entregado_ton"
ID_COLS = ["fecha", "ingenio"]

FEATURES_NOWCASTING = [
    "ingenio", "anio", "mes", "dia_semana", "dia_mes", "semana_anio", "trimestre", "es_fin_de_semana",
    "lluvia_mm", "cana_molida_ton", "lluvia_alta", "temporada_lluviosa", "tiene_comentario_operativo",
    "lluvia_lag_1", "lluvia_lag_7", "lluvia_promedio_7d", "lluvia_promedio_14d",
    "bagazo_lag_1", "bagazo_lag_7", "bagazo_promedio_7d", "cana_lag_1", "cana_promedio_7d"
]

FEATURES_FORECAST = [
    "ingenio", "anio", "mes", "dia_semana", "dia_mes", "semana_anio", "trimestre", "es_fin_de_semana",
    "lluvia_mm", "lluvia_alta", "temporada_lluviosa", "tiene_comentario_operativo",
    "lluvia_lag_1", "lluvia_lag_7", "lluvia_promedio_7d", "lluvia_promedio_14d",
    "bagazo_lag_1", "bagazo_lag_7", "bagazo_promedio_7d", "cana_lag_1", "cana_promedio_7d"
]

SCENARIOS = {
    "A_nowcasting_operativo": {
        "features": FEATURES_NOWCASTING,
        "description": "Incluye caña molida del mismo día. Útil para estimación operativa si la variable está disponible al momento de estimar.",
        "leakage_risk": "medio"
    },
    "B_forecast_sin_leakage": {
        "features": FEATURES_FORECAST,
        "description": "Excluye caña molida del mismo día y usa histórico reciente. Más realista para anticipación.",
        "leakage_risk": "bajo"
    }
}

for name, meta in SCENARIOS.items():
    print(f"\n{name}")
    print(meta["description"])
    print(f"Features: {len(meta['features'])}")
    print(f"Riesgo leakage: {meta['leakage_risk']}")


In [0]:
from pyspark.sql import functions as F

TABLE_FEATURES = "workspace.ml_features.bagazo_features_training"

features_fixed = (
    spark.table(TABLE_FEATURES)
    .withColumn(
        "trimestre",
        F.quarter(F.col("fecha")).cast("int")
    )
    .withColumn(
        "es_fin_de_semana",
        F.when(F.col("dia_semana").isin(1, 7), F.lit(1)).otherwise(F.lit(0)).cast("int")
    )
)

(
    features_fixed
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_FEATURES)
)

print("✅ Tabla de features corregida con trimestre y es_fin_de_semana")

spark.table(TABLE_FEATURES).printSchema()

In [0]:
# ==========================================================
# 6. Preparar datos para scikit-learn con split temporal
# ==========================================================
def prepare_sklearn_dataset(spark_df, feature_cols, target_col=TARGET, test_size=TEST_SIZE):
    """
    Prepara un dataset para scikit-learn conservando la trazabilidad operacional.

    La tabla de features trae columnas de identificación como fecha e ingenio.
    Ingenio también se usa como feature categórica del modelo, por eso la selección
    inicial debe deduplicar columnas antes de convertir a pandas.
    """

    id_cols = list(dict.fromkeys(ID_COLS))
    feature_cols_clean = list(dict.fromkeys(feature_cols))
    required_cols = list(dict.fromkeys(id_cols + feature_cols_clean + [target_col]))

    available_cols = spark_df.columns
    missing_cols = [c for c in required_cols if c not in available_cols]
    if missing_cols:
        raise ValueError(
            "Faltan columnas en la tabla de features: "
            + ", ".join(missing_cols)
            + "Columnas disponibles: "
            + ", ".join(available_cols)
        )

    subset_cols = list(dict.fromkeys(feature_cols_clean + [target_col]))

    pdf = (
        spark_df
        .select(*required_cols)
        .dropna(subset=subset_cols)
        .toPandas()
    )

    pdf["fecha"] = pd.to_datetime(pdf["fecha"])
    pdf = pdf.sort_values(["fecha", "ingenio"]).reset_index(drop=True)

    trace = pdf[id_cols].copy()
    y = pdf[target_col].astype(float).copy()
    X_raw = pdf[feature_cols_clean].copy()

    categorical_cols = [c for c in ["ingenio"] if c in X_raw.columns]
    X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)

    bool_cols = X.select_dtypes(include=["bool"]).columns
    if len(bool_cols) > 0:
        X[bool_cols] = X[bool_cols].astype(int)

    split_index = int(len(X) * (1 - test_size))

    X_train = X.iloc[:split_index].copy()
    X_test = X.iloc[split_index:].copy()
    y_train = y.iloc[:split_index].copy()
    y_test = y.iloc[split_index:].copy()
    trace_train = trace.iloc[:split_index].copy()
    trace_test = trace.iloc[split_index:].copy()

    return X_train, X_test, y_train, y_test, trace_train, trace_test

# Prueba rápida con escenario principal sin leakage
X_train_demo, X_test_demo, y_train_demo, y_test_demo, trace_train_demo, trace_test_demo = prepare_sklearn_dataset(
    spark.table(TABLE_FEATURES), SCENARIOS["B_forecast_sin_leakage"]["features"]
)
print("✅ Dataset preparado")
print("Train:", X_train_demo.shape, "Test:", X_test_demo.shape)
print("Rango train:", trace_train_demo["fecha"].min(), "→", trace_train_demo["fecha"].max())
print("Rango test:", trace_test_demo["fecha"].min(), "→", trace_test_demo["fecha"].max())


## 4. MLflow Tracking

Cada modelo será una carrera. Cada carrera guardará parámetros, métricas, artefactos y el modelo.


In [0]:
# ==========================================================
# 7. Funciones de evaluación y logging MLflow
# ==========================================================
def smape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denominator != 0
    if mask.sum() == 0:
        return 0.0
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denominator[mask]) * 100)


def evaluate_regression(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(mean_squared_error(y_true, y_pred, squared=False)),
        "r2": float(r2_score(y_true, y_pred)),
        "smape": float(smape(y_true, y_pred))
    }


def create_residual_plot(y_true, y_pred, output_path):
    residuals = np.array(y_true) - np.array(y_pred)
    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred, residuals, alpha=0.65)
    plt.axhline(0, linestyle="--")
    plt.title("Residual plot — Bagazo")
    plt.xlabel("Predicción")
    plt.ylabel("Residual")
    plt.tight_layout()
    plt.savefig(output_path, dpi=140)
    plt.close()


def create_actual_vs_predicted_plot(y_true, y_pred, output_path):
    plt.figure(figsize=(8, 5))
    plt.scatter(y_true, y_pred, alpha=0.65)
    min_v = min(np.min(y_true), np.min(y_pred))
    max_v = max(np.max(y_true), np.max(y_pred))
    plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
    plt.title("Actual vs Predicted — Bagazo")
    plt.xlabel("Bagazo real")
    plt.ylabel("Bagazo predicho")
    plt.tight_layout()
    plt.savefig(output_path, dpi=140)
    plt.close()


def create_feature_importance(model, X_test, y_test, feature_names, output_csv, output_png=None):
    importance_df = None
    if hasattr(model, "feature_importances_"):
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)
    elif hasattr(model, "coef_"):
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance": np.abs(model.coef_)
        }).sort_values("importance", ascending=False)
    else:
        try:
            result = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
            importance_df = pd.DataFrame({
                "feature": feature_names,
                "importance": result.importances_mean
            }).sort_values("importance", ascending=False)
        except Exception:
            importance_df = pd.DataFrame({"feature": [], "importance": []})

    importance_df.to_csv(output_csv, index=False)

    if output_png and len(importance_df) > 0:
        top = importance_df.head(12).sort_values("importance", ascending=True)
        plt.figure(figsize=(8, 5))
        plt.barh(top["feature"], top["importance"])
        plt.title("Top feature importance")
        plt.xlabel("Importancia")
        plt.tight_layout()
        plt.savefig(output_png, dpi=140)
        plt.close()

    return importance_df


def build_model_card_text(run_name, scenario_name, model_name, metrics, feature_cols, limitations=None):
    limitations = limitations or []
    return f"""# Model Card — Bagazo MLflow Sesión 8

## Nombre del modelo
{run_name}

## Problema de negocio
Estimar el bagazo entregado por ingenio usando lluvia, caña molida, estacionalidad e histórico reciente.

## Variable objetivo
`target_bagazo_entregado_ton`

## Fuente
`{SOURCE_BAGAZO}` → `{TABLE_FEATURES}`

## Escenario
{scenario_name}

## Modelo
{model_name}

## Métricas holdout
- MAE: {metrics['mae']:.4f}
- RMSE: {metrics['rmse']:.4f}
- R2: {metrics['r2']:.4f}
- SMAPE: {metrics['smape']:.4f}%

## Features principales
{', '.join(feature_cols)}

## Limitaciones
""" + "\n".join([f"- {x}" for x in limitations]) + "\n\n## Siguiente paso MLOps\nConvertir este champion en inferencia batch en la Sesión 9, guardando predicciones y monitoreando error.\n"


def train_and_log_model(scenario_name, scenario_meta, model_name, model, experiment_path):
    feature_cols = scenario_meta["features"]
    X_train, X_test, y_train, y_test, trace_train, trace_test = prepare_sklearn_dataset(
        spark.table(TABLE_FEATURES), feature_cols
    )

    run_name = f"{scenario_name}__{model_name}"
    with mlflow.start_run(run_name=run_name) as run:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        metrics = evaluate_regression(y_test, y_pred)

        params = {
            "modelo": model_name,
            "escenario": scenario_name,
            "target": TARGET,
            "split_strategy": "temporal_80_20",
            "features_count": len(feature_cols),
            "train_rows": int(len(X_train)),
            "test_rows": int(len(X_test)),
            "random_state": RANDOM_STATE,
            "leakage_risk": scenario_meta["leakage_risk"]
        }
        # Agregar hiperparámetros si el modelo los expone
        if hasattr(model, "get_params"):
            for k, v in model.get_params().items():
                if isinstance(v, (str, int, float, bool, type(None))):
                    params[f"hp_{k}"] = v

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags({
            "sesion": "08",
            "caso": "bagazo",
            "tipo": "regresion",
            "fuente_gold": SOURCE_BAGAZO
        })

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = os.path.abspath(tmpdir)
            metrics_path = os.path.join(tmpdir, "metrics_summary.csv")
            pred_path = os.path.join(tmpdir, "predictions_holdout_sample.csv")
            residuals_path = os.path.join(tmpdir, "residuals_plot.png")
            actual_vs_pred_path = os.path.join(tmpdir, "actual_vs_predicted_plot.png")
            importance_path = os.path.join(tmpdir, "feature_importance.csv")
            importance_png_path = os.path.join(tmpdir, "feature_importance_plot.png")
            model_card_path = os.path.join(tmpdir, "model_card_bagazo.md")
            config_path = os.path.join(tmpdir, "training_config.json")

            pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
            pred_df = trace_test.copy()
            pred_df["bagazo_real"] = y_test.values
            pred_df["bagazo_predicho"] = y_pred
            pred_df["error"] = pred_df["bagazo_real"] - pred_df["bagazo_predicho"]
            pred_df["error_absoluto"] = pred_df["error"].abs()
            pred_df["modelo"] = model_name
            pred_df["run_id"] = run.info.run_id
            pred_df["escenario"] = scenario_name
            pred_df["fecha_ejecucion"] = pd.Timestamp.now()
            pred_df.to_csv(pred_path, index=False)

            create_residual_plot(y_test, y_pred, residuals_path)
            create_actual_vs_predicted_plot(y_test, y_pred, actual_vs_pred_path)
            create_feature_importance(model, X_test, y_test, X_train.columns.tolist(), importance_path, importance_png_path)

            model_card = build_model_card_text(
                run_name=run_name,
                scenario_name=scenario_name,
                model_name=model_name,
                metrics=metrics,
                feature_cols=feature_cols,
                limitations=[
                    "Dataset educativo de tamaño reducido en Databricks Free Edition.",
                    "No incorpora variables de humedad, logística, mantenimiento, inventario ni transporte.",
                    "Las ventanas históricas dependen de la calidad del registro por ingenio.",
                    "El escenario nowcasting puede no ser válido si caña molida no está disponible al momento de predecir."
                ]
            )
            open(model_card_path, "w", encoding="utf-8").write(model_card)
            open(config_path, "w", encoding="utf-8").write(json.dumps(params, ensure_ascii=False, indent=2))

            for artifact in [metrics_path, pred_path, residuals_path, actual_vs_pred_path, importance_path, model_card_path, config_path]:
                mlflow.log_artifact(artifact)
            if os.path.exists(importance_png_path):
                mlflow.log_artifact(importance_png_path)

        # Log del modelo. La firma ayuda a documentar entradas/salidas si el entorno lo permite.
        try:
            if infer_signature is not None:
                signature = infer_signature(X_train.head(5), model.predict(X_train.head(5)))
                mlflow.sklearn.log_model(model, artifact_path="model", signature=signature, input_example=X_train.head(5))
            else:
                mlflow.sklearn.log_model(model, artifact_path="model")
        except Exception as e:
            print(f"⚠️ No se pudo inferir firma; se loggea el modelo sin signature. Detalle: {e}")
            mlflow.sklearn.log_model(model, artifact_path="model")

        result = {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "escenario": scenario_name,
            "modelo": model_name,
            "mae": metrics["mae"],
            "rmse": metrics["rmse"],
            "r2": metrics["r2"],
            "smape": metrics["smape"],
            "n_train": int(len(X_train)),
            "n_test": int(len(X_test)),
            "features_count": int(len(feature_cols)),
            "leakage_risk": scenario_meta["leakage_risk"],
            "model_uri": f"runs:/{run.info.run_id}/model",
            "fecha_ejecucion": pd.Timestamp.now()
        }
        return result, pred_df, model


In [0]:
# ==========================================================
# 8. Ejecutar experimento MLflow: baseline vs modelos candidatos
# ==========================================================
def configure_mlflow_experiment(experiment_name="sesion_08_bagazo_mlflow"):
    """
    Configura MLflow Tracking para Databricks.

    En algunos entornos Free Edition / Spark Connect, MLflow puede intentar
    resolver configuraciones internas de Model Registry que no están disponibles.
    Por eso se define explícitamente el tracking URI y el registry URI antes
    de crear o seleccionar el experimento.
    """

    try:
        username = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
        experiment_path = f"/Users/{username}/{experiment_name}"
    except Exception:
        experiment_path = f"/Shared/{experiment_name}"

    try:
        mlflow.set_tracking_uri("databricks")
        mlflow.set_registry_uri("databricks")
        mlflow.set_experiment(experiment_path)

        print(f"✅ Experimento MLflow configurado: {experiment_path}")
        print(f"Tracking URI: {mlflow.get_tracking_uri()}")
        print(f"Registry URI: {mlflow.get_registry_uri()}")

        return experiment_path, "databricks"

    except Exception as e:
        print("⚠️ No fue posible configurar MLflow contra el tracking nativo de Databricks.")
        print("Se usará un tracking local como alternativa para no detener la sesión.")
        print(f"Detalle técnico: {type(e).__name__}: {str(e)[:500]}")

        local_tracking_dir = "/tmp/mlruns_sesion_08_bagazo"
        mlflow.set_tracking_uri(f"file:{local_tracking_dir}")
        mlflow.set_registry_uri(f"file:{local_tracking_dir}")
        mlflow.set_experiment(experiment_name)

        print(f"✅ Experimento MLflow configurado en modo local: {local_tracking_dir}")
        print(f"Tracking URI: {mlflow.get_tracking_uri()}")

        return experiment_name, "local"


EXPERIMENT_PATH, MLFLOW_MODE = configure_mlflow_experiment()

models_to_train = {
    "baseline_dummy_mean": DummyRegressor(strategy="mean"),
    "ridge_regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "random_forest": RandomForestRegressor(
        n_estimators=120,
        max_depth=7,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=1
    )
}

if ENABLE_OPTIONAL_GRADIENT_BOOSTING:
    models_to_train["gradient_boosting"] = GradientBoostingRegressor(random_state=RANDOM_STATE)

results = []
predictions_by_run = {}
models_by_run = {}

for scenario_name, scenario_meta in SCENARIOS.items():
    print(f"\n🚀 Escenario: {scenario_name}")

    for model_name, model in models_to_train.items():
        print(f"Entrenando: {model_name}")

        result, pred_df, fitted_model = train_and_log_model(
            scenario_name=scenario_name,
            scenario_meta=scenario_meta,
            model_name=model_name,
            model=model,
            experiment_path=EXPERIMENT_PATH
        )

        result["mlflow_mode"] = MLFLOW_MODE

        results.append(result)
        predictions_by_run[result["run_id"]] = pred_df
        models_by_run[result["run_id"]] = fitted_model

        print(
            f"✅ {result['run_name']} | "
            f"MAE={result['mae']:.2f} | "
            f"RMSE={result['rmse']:.2f} | "
            f"R2={result['r2']:.3f}"
        )

summary_pdf = (
    pd.DataFrame(results)
    .sort_values(["mae", "rmse"])
    .reset_index(drop=True)
)

display(spark.createDataFrame(summary_pdf))


## TODO 1 — Interpretar MAE

Escribe una frase de negocio: ¿qué significa un MAE de X toneladas para un ingenio?


### Respuesta TODO 1 — Interpretación de MAE

**MAE del champion: 97.01 toneladas**

Desde una perspectiva de negocio, esto significa que el modelo se equivoca en promedio por **97 toneladas** al predecir el bagazo entregado por un ingenio en un día determinado.

Para un ingenio que procesa entre 500-1000 toneladas de caña diaria, un error de ~97 toneladas representa aproximadamente un **10-20% de desviación** en la estimación del bagazo disponible. Esto tiene implicaciones operativas directas:

* **Planificación energética**: El ingenio podría sobreestimar o subestimar la energía que puede generar con el bagazo
* **Gestión de inventario**: Dificulta planificar el almacenamiento o venta de excedentes
* **Decisiones de compra**: Si se subestima, podría necesitar combustible alternativo de emergencia

En términos prácticos, el modelo es útil para **planeación anticipada** (por ejemplo, para el día siguiente), pero las decisiones operativas críticas del mismo día deberían validarse con mediciones reales.

In [0]:
# ==========================================================
# 9. Comparación de experimentos y selección de champion
# ==========================================================
summary_pdf = pd.DataFrame(results).copy()
summary_pdf["rank_mae"] = summary_pdf["mae"].rank(method="dense", ascending=True).astype(int)
summary_pdf["rank_rmse"] = summary_pdf["rmse"].rank(method="dense", ascending=True).astype(int)
summary_pdf["criterio_operativo"] = np.where(summary_pdf["escenario"].str.startswith("B_"), "anticipacion", "estimacion_operativa")
summary_pdf["recomendacion"] = np.where(
    summary_pdf["escenario"].str.startswith("B_"),
    "preferible si el objetivo es anticipar decisiones sin depender de caña del mismo día",
    "útil si caña molida del día está disponible al momento de estimar"
)

# Criterio sugerido: priorizar escenario B si su MAE está dentro del 20% del mejor MAE global; de lo contrario, discutir trade-off.
best_global_mae = summary_pdf["mae"].min()
scenario_b = summary_pdf[summary_pdf["escenario"].str.startswith("B_")].sort_values(["mae", "rmse"])
if len(scenario_b) > 0 and scenario_b.iloc[0]["mae"] <= best_global_mae * 1.20:
    champion = scenario_b.iloc[0].to_dict()
else:
    champion = summary_pdf.sort_values(["mae", "rmse"]).iloc[0].to_dict()

summary_pdf["is_champion"] = summary_pdf["run_id"] == champion["run_id"]
summary_spark = spark.createDataFrame(summary_pdf)
summary_spark.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(TABLE_SUMMARY)

print(f"✅ Tabla resumen creada: {TABLE_SUMMARY}")
print("🏆 Champion sugerido")
print(json.dumps({k: str(v) for k, v in champion.items()}, ensure_ascii=False, indent=2))

display(spark.table(TABLE_SUMMARY).orderBy(F.desc("is_champion"), F.asc("mae")))


## TODO 2 — Elegir champion

Revisa la tabla comparativa y justifica si estás de acuerdo con el champion sugerido. Considera MAE, RMSE, interpretabilidad y leakage.


In [0]:
# ==========================================================
# 10. Modelo champion: registro empresarial y fallback académico
# ==========================================================
"""
En esta sesión el modelo champion ya quedó loggeado dentro del run de MLflow.
Ese artifact es suficiente para la práctica de Free Edition: permite revisar el modelo,
sus métricas, sus parámetros y sus artefactos desde MLflow Tracking.

El registro formal en Model Registry / Models in Unity Catalog es un paso de gobierno
empresarial. En un entorno productivo de Azure Databricks normalmente requiere:

- Unity Catalog correctamente configurado.
- Permisos sobre el catálogo y el schema.
- Permiso para crear modelos.
- Permisos de escritura sobre el storage administrado del catálogo.
- Política de gobierno para versionamiento, aliases y promoción de modelos.

Por eso, en Free Edition dejamos el modelo como artifact de MLflow y registramos
la decisión en una tabla de auditoría. Si el instructor está en un workspace empresarial,
puede activar ENABLE_UC_MODEL_REGISTRY = True.
"""

registry_rows = []

model_uri = champion["model_uri"]
registered_model_name_uc = f"{SCHEMA_MODELS}.bagazo_champion_sesion_08"

status = "artifact_only_free_edition"
message = (
    "El modelo champion se conserva como artifact del run de MLflow. "
    "El registro formal en Unity Catalog se deja como paso empresarial, "
    "porque en Free Edition o workspaces sin permisos de storage administrado "
    "puede fallar por permisos de catálogo, schema o ubicación administrada."
)
registered_name_used = registered_model_name_uc

if ENABLE_UC_MODEL_REGISTRY:
    try:
        print(f"Intentando registrar modelo en Unity Catalog como: {registered_model_name_uc}")

        previous_registry_uri = mlflow.get_registry_uri()
        mlflow.set_registry_uri("databricks-uc")

        registered_model = mlflow.register_model(
            model_uri=model_uri,
            name=registered_model_name_uc
        )

        status = "registered_uc"
        message = (
            f"Modelo registrado correctamente en Unity Catalog: "
            f"{registered_model.name}, versión {registered_model.version}."
        )

        print("✅", message)

        try:
            mlflow.set_registry_uri(previous_registry_uri)
        except Exception:
            pass

    except Exception as e:
        raw_message = str(e)

        if "AccessDenied" in raw_message or "PutObject" in raw_message:
            status = "fallback_artifact_only_storage_permission"
            message = (
                "No fue posible registrar el modelo en Unity Catalog porque el entorno "
                "no tiene permisos suficientes para escribir los artefactos del modelo "
                "en el storage administrado del catálogo. El modelo se conserva como "
                "artifact dentro del run de MLflow."
            )
        elif "PERMISSION_DENIED" in raw_message:
            status = "fallback_artifact_only_permission_denied"
            message = (
                "No fue posible registrar el modelo por permisos insuficientes sobre "
                "Model Registry / Unity Catalog. El modelo se conserva como artifact "
                "dentro del run de MLflow."
            )
        else:
            status = "fallback_artifact_only_registry_error"
            message = (
                "No fue posible registrar el modelo en el registry del workspace. "
                "El modelo se conserva como artifact dentro del run de MLflow. "
                f"Detalle técnico resumido: {raw_message[:500]}"
            )

        print("⚠️ Registro formal no disponible en este entorno.")
        print(message)

        try:
            mlflow.set_registry_uri("databricks")
        except Exception:
            pass

else:
    print("✅ Modelo champion conservado como artifact en MLflow.")
    print("ℹ️ Registro formal omitido para mantener compatibilidad con Free Edition.")
    print("ℹ️ En Azure Databricks empresarial, este paso se activaría con Unity Catalog y permisos adecuados.")

registry_rows.append({
    "run_id": champion["run_id"],
    "model_uri": model_uri,
    "registered_name_attempted": registered_name_used,
    "status": status,
    "message": message,
    "fecha_ejecucion": pd.Timestamp.now()
})

registry_log_pdf = pd.DataFrame(registry_rows)

(
    spark.createDataFrame(registry_log_pdf)
    .write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_REGISTRY_LOG)
)

print(f"✅ Log de registro creado: {TABLE_REGISTRY_LOG}")
display(spark.table(TABLE_REGISTRY_LOG))


### Respuesta TODO 2 — Justificación del champion

**Sí, estoy de acuerdo con la elección del champion: B_forecast_sin_leakage__random_forest**

**Análisis comparativo:**

| Aspecto | Champion (B_forecast) | Mejor MAE (A_nowcasting) |
|---------|----------------------|-------------------------|
| MAE | 97.01 ton | 84.71 ton ✓ |
| RMSE | 132.89 ton | 123.92 ton ✓ |
| R2 | 0.7327 | 0.7676 ✓ |
| Leakage risk | **bajo** ✓ | medio |
| Disponibilidad | Sin datos del mismo día ✓ | Requiere caña molida del día |

**Razones para preferir el champion:**

1. **Viabilidad operativa**: El modelo B puede ejecutarse **antes** de iniciar operaciones del día, usando solo datos históricos y pronóstico de lluvia. El modelo A necesita datos que solo están disponibles **durante** el día.

2. **Diferencia aceptable**: La diferencia en MAE es de ~12 toneladas (14% más error), pero está dentro del umbral del 20% establecido en el criterio de selección.

3. **Interpretabilidad**: Random Forest es más interpretable que modelos de caja negra y permite entender qué factores históricos y climáticos influyen.

4. **Riesgo de leakage**: El modelo A tiene riesgo medio porque usa `cana_molida_ton_mismo_dia`, una variable fuertemente correlacionada con el target que no estará disponible al momento de predecir en producción.

**Conclusión**: Sacrificar ~12 toneladas de precisión a cambio de un modelo **deployable** y **sin leakage** es la decisión correcta desde MLOps.

In [0]:
# ==========================================================
# 11. Guardar predicciones holdout del champion
# ==========================================================
champion_predictions = predictions_by_run[champion["run_id"]].copy()
champion_predictions["fecha"] = pd.to_datetime(champion_predictions["fecha"]).dt.date
champion_predictions["fecha_ejecucion"] = pd.to_datetime(champion_predictions["fecha_ejecucion"])

spark.createDataFrame(champion_predictions).write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(TABLE_HOLDOUT)
print(f"✅ Predicciones holdout creadas: {TABLE_HOLDOUT}")
display(spark.table(TABLE_HOLDOUT).orderBy("fecha", "ingenio").limit(20))


In [0]:
# ==========================================================
# 12. Guía para explorar la interfaz MLflow y leer resultados
# ==========================================================
registry_status = spark.table(TABLE_REGISTRY_LOG).select("status").collect()[0]["status"]

print(f"""
🔎 Demo sugerida en la UI de MLflow

1. Abre el panel izquierdo de Databricks.
2. Entra a Experiments.
3. Busca el experimento de la sesión:
   {EXPERIMENT_PATH}
4. Compara los runs por MAE, RMSE, R2 y SMAPE.
5. Abre el run champion.
6. Revisa:
   - Parameters: configuración del entrenamiento.
   - Metrics: desempeño del modelo.
   - Artifacts: plots, CSVs, model card y configuración.
   - Model: modelo sklearn loggeado como artifact reutilizable.

📌 Lectura del champion observado

- MAE: {champion['mae']:.2f} toneladas.
  Significa que el error absoluto promedio del modelo es cercano a ese valor.

- RMSE: {champion['rmse']:.2f} toneladas.
  Penaliza más los errores grandes; si es bastante mayor que MAE, hay días difíciles o extremos.

- R2: {champion['r2']:.3f}.
  Indica qué tanto de la variabilidad del holdout logra capturar el modelo.

- SMAPE: {champion['smape']:.2f}%.
  Debe interpretarse con cuidado cuando existen días con valores reales bajos.

📌 Estado del registro del modelo

Estado actual: {registry_status}

En Free Edition es suficiente que el modelo quede loggeado como artifact del run de MLflow.
En un entorno empresarial de Azure Databricks, el siguiente paso sería registrarlo en Unity Catalog,
asignarle un alias como Champion y usarlo como entrada para inferencia batch o serving.

Mensaje clave: MLflow no es solo guardar números. Es dejar evidencia reproducible del experimento.
""")


## TODO 3 — Conclusión sobre leakage

¿Por qué un modelo con mejor MAE puede ser menos adecuado si usa variables que no están disponibles al momento de predecir?


### Respuesta TODO 3 — Problema del leakage

**Un modelo con mejor MAE puede ser inadecuado si usa variables no disponibles al momento de predecir por las siguientes razones:**

#### 1. **Imposibilidad de deployment en producción**
Si el modelo requiere `cana_molida_ton_mismo_dia` para hacer una predicción, pero esa variable solo se conoce **al final del día**, el modelo no puede usarse para:
* Planificación matutina de operaciones
* Decisiones anticipadas de compra de combustible alternativo
* Coordinación logística con compradores de excedentes

#### 2. **Falla silenciosa en inferencia**
En producción, si falta la variable, el modelo:
* Podría usar valores por defecto incorrectos (ej: 0 o el promedio histórico)
* Generaría predicciones erróneas sin que el usuario lo note
* Degradaría su desempeño real muy por debajo del MAE reportado en holdout

#### 3. **Sobreajuste a correlaciones espurias**
Variables con leakage tienen correlaciones artificialmente altas porque "conocen el futuro". El modelo aprende patrones que no son causales sino consecuencias del proceso:
* `bagazo_entregado` ≈ 0.28 × `cana_molida` (relación física conocida)
* El modelo "aprende" esta regla en lugar de patrones climáticos y operativos verdaderamente predictivos

#### 4. **Ejemplo concreto de este notebook**
* **Modelo A** (MAE 84.71): "Si mañana se muelen 3000 toneladas de caña, habrá ~840 toneladas de bagazo"
  * ❌ **Problema**: No sabemos cuánta caña se molerá hasta que ocurra
* **Modelo B** (MAE 97.01): "Dada la lluvia pronosticada y el bagazo de los últimos 7 días, habrá ~850 toneladas"
  * ✓ **Ventaja**: Todos los datos están disponibles de antemano

**Conclusión**: En ML aplicado, **un modelo deployable con error aceptable vale más que un modelo perfecto en papel pero inutilizable en producción**. El MAE de holdout solo importa si el modelo puede ejecutarse en las condiciones reales de inferencia.

## Mini caso Lumi opcional: mala experiencia del cliente

Este bloque queda como conversación o reto opcional. No desplaza el caso principal de Bagazo.

**Pregunta:** ¿podríamos predecir una mala experiencia del cliente?

**Fuente sugerida:** `workspace.lumi_gold.fact_delivery_experience`  
**Target sugerido:** `mala_experiencia = review_score <= 2`

No se usa pagos como caso principal porque en la Sesión 7 se observó `valor_pagado_total = 0.0` en `kpi_payment_methods`.


In [0]:
# ==========================================================
# 13. Mini caso Lumi opcional (no ejecutar si el tiempo es corto)
# ==========================================================
LUMI_DELIVERY = f"{CATALOG}.lumi_gold.fact_delivery_experience"
if table_exists(LUMI_DELIVERY):
    lumi = spark.table(LUMI_DELIVERY)
    print("✅ Fuente Lumi disponible")
    print(lumi.columns)
    # Ejemplo conceptual: ajustar nombres si difieren.
    # target sugerido: mala_experiencia = review_score <= 2
else:
    print("⚠️ No se encontró fact_delivery_experience. Saltar mini caso Lumi.")


In [0]:
# ==========================================================
# MINI CASO LUMI: Predicción de Mala Experiencia del Cliente
# ==========================================================

print("📊 Explorando datos de experiencia de entrega Lumi\n")

# Ver estadísticas básicas
print(f"Total de registros: {lumi.count():,}")
print(f"\nColumnas disponibles:")
for col in lumi.columns:
    print(f"  - {col}")

# Analizar distribución de review scores
print("\n📈 Distribución de review_score_avg:")
lumi.groupBy("review_score_avg").count().orderBy("review_score_avg").show()

print("\n📈 Distribución de review_score_min:")
lumi.groupBy("review_score_min").count().orderBy("review_score_min").show()

print("\n📈 Análisis de entregas tardías:")
lumi.groupBy("is_late").count().show()

print("\n📈 Distribución de tramo_demora:")
lumi.groupBy("tramo_demora").count().orderBy("tramo_demora").show()

In [0]:
# ==========================================================
# Preparación de datos para clasificación
# ==========================================================
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

# Crear target: mala_experiencia (review_score_min <= 2 o review_score_avg <= 2)
lumi_ml = lumi.filter(F.col("review_records") > 0)  # Solo pedidos con reviews

lumi_ml = lumi_ml.withColumn(
    "mala_experiencia",
    F.when(
        (F.col("review_score_min") <= 2) | (F.col("review_score_avg") <= 2),
        1
    ).otherwise(0).cast(IntegerType())
)

# Crear features adicionales
lumi_ml = lumi_ml.withColumn(
    "delay_ratio",
    F.when(F.col("delivery_days") > 0, F.col("delay_days") / F.col("delivery_days")).otherwise(0.0)
)

lumi_ml = lumi_ml.withColumn(
    "has_delay",
    F.when(F.col("delay_days") > 0, 1).otherwise(0).cast(IntegerType())
)

# Seleccionar columnas base
base_cols = [
    "order_id",
    "customer_id",
    "purchase_date",
    "mala_experiencia",
    "delivery_days",
    "delay_days",
    "delay_ratio",
    "has_delay",
    "is_late",
    "has_review_comment",
    "order_status",
    "tramo_demora"
]

lumi_ml = lumi_ml.select(*base_cols).na.drop()

print(f"\n✅ Dataset preparado: {lumi_ml.count():,} registros")
print(f"\n📊 Distribución del target:")
lumi_ml.groupBy("mala_experiencia").count().show()

# Calcular proporción de clase positiva
total = lumi_ml.count()
malas = lumi_ml.filter(F.col("mala_experiencia") == 1).count()
print(f"\n📈 Proporción de mala experiencia: {malas/total*100:.2f}%")

# Convertir a Pandas para modelado con sklearn
lumi_pandas = lumi_ml.toPandas()

# Codificar variables categóricas con LabelEncoder
from sklearn.preprocessing import LabelEncoder

le_status = LabelEncoder()
le_tramo = LabelEncoder()

lumi_pandas['order_status_idx'] = le_status.fit_transform(lumi_pandas['order_status'])
lumi_pandas['tramo_demora_idx'] = le_tramo.fit_transform(lumi_pandas['tramo_demora'])

# Convertir booleano a int
lumi_pandas['is_late'] = lumi_pandas['is_late'].astype(int)

# Definir columnas de features
feature_cols = [
    "delivery_days",
    "delay_days",
    "delay_ratio",
    "has_delay",
    "is_late",
    "has_review_comment",
    "order_status_idx",
    "tramo_demora_idx"
]

print(f"\n✅ Convertido a Pandas y codificado: {len(lumi_pandas):,} registros")
print(f"\nCategorías de order_status: {list(le_status.classes_)}")
print(f"Categorías de tramo_demora: {list(le_tramo.classes_)}")

display(lumi_pandas.head(10))

In [0]:
# ==========================================================
# Train/Test Split estratificado
# ==========================================================
from sklearn.model_selection import train_test_split

# Separar features y target
X = lumi_pandas[feature_cols].values
y = lumi_pandas["mala_experiencia"].values

print(f"Shape de X: {X.shape}")
print(f"Shape de y: {y.shape}")
print(f"\nDistribución de clases:")
print(f"  Buena experiencia (0): {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
print(f"  Mala experiencia (1): {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"\n✅ Train set: {len(X_train):,} registros")
print(f"✅ Test set: {len(X_test):,} registros")
print(f"\nDistribución en Train:")
print(f"  Clase 0: {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Clase 1: {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
print(f"\nDistribución en Test:")
print(f"  Clase 0: {(y_test==0).sum():,} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  Clase 1: {(y_test==1).sum():,} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")

In [0]:
# ==========================================================
# Funciones de evaluación para clasificación
# ==========================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

def evaluate_classification(y_true, y_pred, y_pred_proba=None):
    """
    Evalúa modelo de clasificación binaria.
    """
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0))
    }
    
    if y_pred_proba is not None:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_pred_proba))
    
    return metrics


def create_confusion_matrix_plot(y_true, y_pred, output_path):
    """
    Crea matriz de confusión.
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Buena', 'Mala'],
                yticklabels=['Buena', 'Mala'])
    plt.title('Matriz de Confusión')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Etiqueta Predicha')
    plt.tight_layout()
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"✅ Matriz de confusión guardada: {output_path}")


def train_and_log_classifier(
    scenario_name,
    model_name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    feature_names,
    experiment_path
):
    """
    Entrena y registra un clasificador en MLflow.
    """
    import tempfile
    import os
    
    run_name = f"{scenario_name}__{model_name}"
    
    with mlflow.start_run(run_name=run_name) as run:
        # Entrenar
        model.fit(X_train, y_train)
        
        # Predecir
        y_pred = model.predict(X_test)
        
        # Predecir probabilidades si el modelo lo soporta
        y_pred_proba = None
        if hasattr(model, "predict_proba"):
            y_pred_proba = model.predict_proba(X_test)[:, 1]
        
        # Evaluar
        metrics = evaluate_classification(y_test, y_pred, y_pred_proba)
        
        # Log parameters
        mlflow.log_param("scenario", scenario_name)
        mlflow.log_param("model_type", model_name)
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("n_features", X_train.shape[1])
        mlflow.log_param("random_state", RANDOM_STATE)
        
        if hasattr(model, "get_params"):
            for param_name, param_value in model.get_params().items():
                try:
                    mlflow.log_param(f"model_{param_name}", param_value)
                except Exception:
                    pass
        
        # Log metrics
        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)
        
        # Crear y guardar matriz de confusión
        with tempfile.TemporaryDirectory() as tmpdir:
            cm_path = os.path.join(tmpdir, "confusion_matrix.png")
            create_confusion_matrix_plot(y_test, y_pred, cm_path)
            mlflow.log_artifact(cm_path, "plots")
        
        # Log classification report
        report = classification_report(y_test, y_pred, 
                                      target_names=['Buena', 'Mala'],
                                      output_dict=False)
        with tempfile.TemporaryDirectory() as tmpdir:
            report_path = os.path.join(tmpdir, "classification_report.txt")
            with open(report_path, 'w') as f:
                f.write(report)
            mlflow.log_artifact(report_path)
        
        # Log model
        mlflow.sklearn.log_model(model, "model")
        
        # Preparar resultado
        result = {
            "run_id": run.info.run_id,
            "run_name": run_name,
            "scenario": scenario_name,
            "model": model_name,
            **metrics,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": X_train.shape[1],
            "model_uri": f"runs:/{run.info.run_id}/model",
            "fecha_ejecucion": pd.Timestamp.now()
        }
        
        return result, model

print("✅ Funciones de evaluación de clasificación definidas")

In [0]:
# ==========================================================
# Entrenar modelos de clasificación con MLflow
# ==========================================================
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Configurar experimento MLflow para Lumi
try:
    username = spark.sql("SELECT current_user() AS user").collect()[0]["user"]
    lumi_experiment_path = f"/Users/{username}/sesion_08_lumi_mala_experiencia"
except Exception:
    lumi_experiment_path = "/Shared/sesion_08_lumi_mala_experiencia"

mlflow.set_experiment(lumi_experiment_path)
print(f"✅ Experimento MLflow configurado: {lumi_experiment_path}\n")

# Modelos a entrenar
classifiers = {
    "baseline_dummy_most_frequent": DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE),
    "baseline_dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        class_weight="balanced"  # Importante para clases desbalanceadas
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=1
    )
}

lumi_results = []
lumi_models = {}

print("🚀 Entrenando modelos de clasificación para Lumi\n")

for model_name, model in classifiers.items():
    print(f"Entrenando: {model_name}")
    
    result, fitted_model = train_and_log_classifier(
        scenario_name="lumi_mala_experiencia",
        model_name=model_name,
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_cols,
        experiment_path=lumi_experiment_path
    )
    
    lumi_results.append(result)
    lumi_models[result["run_id"]] = fitted_model
    
    print(
        f"✅ {result['run_name']} | "
        f"Accuracy={result['accuracy']:.3f} | "
        f"Precision={result['precision']:.3f} | "
        f"Recall={result['recall']:.3f} | "
        f"F1={result['f1_score']:.3f}"
    )
    if "roc_auc" in result:
        print(f"   ROC-AUC={result['roc_auc']:.3f}")
    print()

# Crear tabla resumen
lumi_summary = pd.DataFrame(lumi_results).sort_values(["f1_score", "roc_auc"], ascending=False).reset_index(drop=True)

print("\n📊 Resumen de modelos Lumi:\n")
display(spark.createDataFrame(lumi_summary))

In [0]:
# ==========================================================
# Selección de champion para Lumi
# ==========================================================

# Criterio: priorizar F1-score (balance entre precision y recall)
# En casos desbalanceados, F1 es mejor que accuracy

lumi_summary_copy = lumi_summary.copy()
lumi_summary_copy["rank_f1"] = lumi_summary_copy["f1_score"].rank(method="dense", ascending=False).astype(int)
lumi_summary_copy["rank_precision"] = lumi_summary_copy["precision"].rank(method="dense", ascending=False).astype(int)
lumi_summary_copy["rank_recall"] = lumi_summary_copy["recall"].rank(method="dense", ascending=False).astype(int)

# Seleccionar champion basado en F1 score
lumi_champion = lumi_summary_copy.sort_values(["f1_score", "roc_auc"], ascending=False).iloc[0].to_dict()
lumi_summary_copy["is_champion"] = lumi_summary_copy["run_id"] == lumi_champion["run_id"]

# Guardar en tabla Delta
LUMI_TABLE_SUMMARY = f"{SCHEMA_MODELS}.lumi_experiment_summary_sesion_08"

(
    spark.createDataFrame(lumi_summary_copy)
    .write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable(LUMI_TABLE_SUMMARY)
)

print(f"✅ Tabla resumen creada: {LUMI_TABLE_SUMMARY}\n")
print("🏆 Champion Lumi seleccionado:\n")
print(json.dumps({k: str(v) if not isinstance(v, (int, float, bool)) else v 
                  for k, v in lumi_champion.items()}, 
                 ensure_ascii=False, indent=2))

print("\n📊 Tabla completa ordenada por F1-score:\n")
display(spark.table(LUMI_TABLE_SUMMARY).orderBy(F.desc("is_champion"), F.desc("f1_score")))

## Análisis del Mini Caso Lumi - Predicción de Mala Experiencia

### Métricas Clave para Clasificación

**Contexto del problema**: Predecir si un cliente tendrá una mala experiencia (review score ≤ 2) basándose en datos de entrega.

#### Interpretación de métricas:

* **Accuracy**: Proporción de predicciones correctas. Puede ser engañosa en datasets desbalanceados.

* **Precision**: De todos los casos que predijimos como "mala experiencia", ¿cuántos realmente lo fueron?
  * Alta precisión = pocos falsos positivos
  * Importante si el costo de actuar innecesariamente es alto

* **Recall (Sensibilidad)**: De todos los casos reales de "mala experiencia", ¿cuántos detectamos?
  * Alto recall = pocos falsos negativos
  * Importante si no queremos dejar pasar casos problemáticos

* **F1-Score**: Media armónica entre precision y recall
  * Útil cuando necesitamos balance entre ambas métricas
  * Especialmente relevante en datasets desbalanceados

* **ROC-AUC**: Mide la capacidad del modelo de distinguir entre clases
  * Valor de 0.5 = modelo aleatorio
  * Valor de 1.0 = modelo perfecto

### Trade-offs del Negocio

**Si priorizamos Recall** (detectar todas las malas experiencias):
* ✓ No se nos escapan clientes insatisfechos
* ✓ Podemos intervenir proactivamente (descuentos, compensaciones)
* ✗ Gastaremos recursos en falsos positivos

**Si priorizamos Precision** (solo actuar cuando estamos seguros):
* ✓ Eficiencia en uso de recursos (descuentos bien dirigidos)
* ✓ No molestamos a clientes satisfechos con encuestas/compensaciones
* ✗ Dejamos pasar casos reales de insatisfacción

**F1-Score busca el equilibrio** entre ambos extremos.

## Conclusiones del Mini Caso Lumi

### Champion Seleccionado: Logistic Regression

**Métricas del modelo:**
* **Accuracy**: 64.9% (no es la métrica más relevante en datasets desbalanceados)
* **Precision**: 24.7% (de cada 4 alertas, solo 1 es correcta)
* **Recall**: 84.6% (✓ detecta 85% de las malas experiencias reales)
* **F1-Score**: 0.382 (balance entre precision y recall)
* **ROC-AUC**: 0.807 (✓ buena capacidad para distinguir entre clases)

### Interpretación de Negocio

#### Lo Bueno ✓
1. **Alto Recall (84.6%)**: El modelo detecta la mayoría de los clientes insatisfechos
   * Solo se nos escapan ~15% de las malas experiencias
   * Podemos intervenir proactivamente antes de que el cliente deje una reseña negativa

2. **ROC-AUC alto (0.807)**: El modelo tiene buena capacidad de discriminación
   * Puede rankear clientes por probabilidad de insatisfacción
   * Útil para priorizar intervenciones según presupuesto

3. **Simplicidad**: Logistic Regression es interpretable
   * Podemos ver qué factores aumentan el riesgo de mala experiencia
   * Fácil de explicar a stakeholders

#### El Desafío
1. **Baja Precision (24.7%)**: 3 de cada 4 alertas son falsas alarmas
   * Si enviamos compensaciones automáticas, desperdiciaremos ~75% del presupuesto
   * Si enviamos encuestas, molestaremos a clientes satisfechos

### Estrategias de Deployment

#### Opción 1: Sistema de Alerta con Revisión Humana
* El modelo genera lista de pedidos en riesgo
* Equipo de CS revisa manualmente antes de intervenir
* **Costo**: Tiempo de equipo humano
* **Beneficio**: No se desperdician compensaciones

#### Opción 2: Intervenciones de Bajo Costo
* Enviar mensaje proactivo: "¿Todo bien con tu pedido?"
* Email con tracking mejorado
* Notificación de disponibilidad de soporte
* **Costo**: Bajo (automatizado)
* **Beneficio**: Mejora experiencia sin costo significativo por falsos positivos

#### Opción 3: Sistema de Scoring con Umbrales
* Usar `predict_proba` para generar score 0-100%
* Umbral alto (ej: >70% probabilidad) → intervención directa
* Umbral medio (40-70%) → monitoreo
* Umbral bajo (<40%) → flujo normal
* **Costo**: Medio
* **Beneficio**: Balance entre cobertura y eficiencia

### Mejoras Futuras

1. **Más features**:
   * Historial de pedidos del cliente
   * Tipo de producto / categoría
   * Ubicación geográfica
   * Transportadora asignada
   * Distancia de envío

2. **Ingeniería de features**:
   * Tendencia de demoras del mismo transportista
   * Interacciones: demora × tipo_producto
   * Features temporales: día de la semana, temporada

3. **Ajuste de umbral**:
   * Actualmente usamos 0.5 por defecto
   * Podríamos bajar a 0.3 para aumentar recall (detectar más casos)
   * O subir a 0.7 para aumentar precision (menos falsas alarmas)

4. **Cost-sensitive learning**:
   * Penalizar más los falsos negativos (clientes insatisfechos no detectados)
   * Ajustar `class_weight` para reflejar costos reales de negocio

### Valor de Negocio

**Escenario de ejemplo:**
* 1,000 pedidos diarios
* 128 malas experiencias reales (12.8%)
* El modelo detecta ~108 de ellas (84.6% recall)
* Genera ~440 alertas totales (precision 24.7%)

**Si cada compensación cuesta $10 y cada cliente perdido vale $100:**
* **Sin modelo**: Pierdes 128 clientes = $12,800 en valor de vida
* **Con modelo + revisión humana**: Salvas 108 clientes = $10,800 recuperados
* **ROI neto**: ~$10,800 - (costo de revisión) = Valor positivo

**Conclusión**: Incluso con baja precision, el modelo agrega valor si se implementa con la estrategia correcta.

## TODO 4 — Variable adicional de negocio

Propón una variable que mejoraría el modelo: humedad, mantenimiento, transporte, inventario, demanda energética u otra.


### Respuesta TODO 4 — Variable Adicional de Negocio

## Variable Propuesta: **Humedad Relativa del Bagazo (%)**

### Justificación Técnica

El bagazo es un subproducto de la molienda de caña que contiene agua residual. La **humedad del bagazo** afecta directamente:

1. **Peso entregado**: Bagazo más húmedo pesa más (agua añade masa)
2. **Eficiencia energética**: Mayor humedad reduce el poder calorífico
3. **Capacidad de almacenamiento**: Bagazo húmedo se compacta más

### Relación con Variables Existentes

**Correlaciones esperadas:**
* **Lluvia (mm)** → **Humedad del bagazo** → **Bagazo entregado (ton)**
  * Días lluviosos: Caña más húmeda → bagazo más húmedo → mayor peso
  * La lluvia actual solo captura el clima, no el estado físico del bagazo

* **Caña molida (ton)** + **Humedad (%)** → Mejor predicción de bagazo
  * Actualmente: `bagazo ≈ 0.28 × caña_molida` (relación física)
  * Con humedad: `bagazo ≈ (0.28 × caña_molida) × (1 + factor_humedad)`



## TODO 5 — Regla de monitoreo para Sesión 9

Diseña una regla simple: por ejemplo, alertar si el MAE semanal supera X toneladas o si el error absoluto promedio crece 20%.


### Respuesta TODO 5 — Reglas de Monitoreo para Sesión 9

## Sistema de Alertas para Degradación del Modelo

### Objetivo del Monitoreo

Detectar cuándo el modelo champion (Random Forest, MAE=97.01 ton) **deja de ser confiable** debido a:
* **Data drift**: Distribución de features cambia (ej: nuevas variedades de caña)
* **Concept drift**: Relación caña→bagazo cambia (ej: mejoras en molienda)
* **Calidad de datos**: Sensores descalibrados, valores faltantes
* **Eventos excepcionales**: Mantenimiento mayor, cambios climáticos extremos

---

## Regla 1: Alerta por MAE Semanal Elevado

### Configuración
```python
# Umbrales
MAE_BASELINE = 97.01  # MAE del champion en holdout
MAE_WARNING_THRESHOLD = 116.4  # +20% del baseline
MAE_CRITICAL_THRESHOLD = 135.8  # +40% del baseline
WINDOW_DAYS = 7  # Ventana de evaluación
```

### Lógica de Alerta
```sql
-- Query diaria para calcular MAE semanal
WITH weekly_errors AS (
  SELECT 
    fecha,
    ingenio,
    bagazo_real,
    bagazo_predicho,
    ABS(bagazo_real - bagazo_predicho) AS error_absoluto,
    AVG(ABS(bagazo_real - bagazo_predicho)) 
      OVER (ORDER BY fecha ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS mae_7d
  FROM workspace.ml_monitoring.bagazo_predictions_production
  WHERE fecha >= CURRENT_DATE - INTERVAL 7 DAYS
)
SELECT 
  CURRENT_DATE AS fecha_alerta,
  'MAE Semanal Elevado' AS tipo_alerta,
  ROUND(AVG(mae_7d), 2) AS mae_semanal_actual,
  97.01 AS mae_baseline,
  ROUND((AVG(mae_7d) - 97.01) / 97.01 * 100, 1) AS pct_degradacion,
  CASE 
    WHEN AVG(mae_7d) >= 135.8 THEN 'CRÍTICO'
    WHEN AVG(mae_7d) >= 116.4 THEN 'WARNING'
    ELSE 'OK'
  END AS severidad
FROM weekly_errors
HAVING AVG(mae_7d) >= 116.4;
```

### Acciones por Severidad

**WARNING (MAE 116-135 ton, +20-40%)**
* Email a equipo de Data Science
* Revisar distribución de features en última semana
* Investigar ingenios con mayor error
* Programar re-entrenamiento en 2 semanas si persiste

**CRÍTICO (MAE >135 ton, +40%)**
* Alerta inmediata (Slack + PagerDuty)
* Pausar uso del modelo en producción (revertir a baseline dummy)
* Investigación de causa raíz en 24h
* Re-entrenamiento de emergencia con datos recientes

---

## Regla 2: Alerta por Crecimiento Sostenido del Error

### Configuración
```python
# Detectar tendencia creciente (no solo picos aislados)
GROWTH_THRESHOLD = 0.20  # 20% de crecimiento
COMPARISON_WINDOW = 14  # Comparar últimas 2 semanas vs 2 semanas anteriores
MIN_PREDICTIONS = 50  # Mínimo de predicciones para activar regla
```

### Lógica de Alerta
```sql
-- Detectar tendencia de degradación gradual
WITH mae_by_period AS (
  SELECT 
    'Semanas 3-4' AS periodo,
    AVG(ABS(bagazo_real - bagazo_predicho)) AS mae_promedio,
    COUNT(*) AS n_predicciones
  FROM workspace.ml_monitoring.bagazo_predictions_production
  WHERE fecha BETWEEN CURRENT_DATE - INTERVAL 28 DAYS AND CURRENT_DATE - INTERVAL 14 DAYS
  
  UNION ALL
  
  SELECT 
    'Semanas 1-2' AS periodo,
    AVG(ABS(bagazo_real - bagazo_predicho)) AS mae_promedio,
    COUNT(*) AS n_predicciones
  FROM workspace.ml_monitoring.bagazo_predictions_production
  WHERE fecha >= CURRENT_DATE - INTERVAL 14 DAYS
),
comparison AS (
  SELECT 
    MAX(CASE WHEN periodo = 'Semanas 3-4' THEN mae_promedio END) AS mae_anterior,
    MAX(CASE WHEN periodo = 'Semanas 1-2' THEN mae_promedio END) AS mae_reciente,
    SUM(n_predicciones) AS total_predicciones
  FROM mae_by_period
)
SELECT 
  CURRENT_DATE AS fecha_alerta,
  'Degradación Gradual' AS tipo_alerta,
  ROUND(mae_anterior, 2) AS mae_sem_3_4,
  ROUND(mae_reciente, 2) AS mae_sem_1_2,
  ROUND((mae_reciente - mae_anterior) / mae_anterior * 100, 1) AS pct_crecimiento,
  CASE 
    WHEN (mae_reciente - mae_anterior) / mae_anterior >= 0.30 THEN 'CRÍTICO'
    WHEN (mae_reciente - mae_anterior) / mae_anterior >= 0.20 THEN 'WARNING'
    ELSE 'OK'
  END AS severidad
FROM comparison
WHERE (mae_reciente - mae_anterior) / mae_anterior >= 0.20
  AND total_predicciones >= 50;
```

### Acciones por Severidad

**WARNING (+20-30% crecimiento)**
* Análisis de features drift (comparar distribuciones)
* Revisar calidad de datos de entrada
* Evaluar si cambios operativos explican la tendencia

**CRÍTICO (+30% o más)**
* Re-entrenamiento inmediato con datos recientes
* Validar si el concepto cambió (nueva relación caña→bagazo)
* Considerar agregar features adicionales (ej: humedad)

---

##  Regla 3: Alerta por Predicciones Fuera de Rango

### Configuración
```python
# Detectar predicciones físicamente imposibles o extremas
BAGAZO_MIN_FISICO = 0  # No puede ser negativo
BAGAZO_MAX_PERCENTIL_99 = 850  # Basado en histórico
OUTLIER_THRESHOLD = 0.05  # Alertar si >5% de predicciones son outliers
```

### Lógica de Alerta
```sql
-- Detectar predicciones anómalas
WITH daily_outliers AS (
  SELECT 
    fecha,
    COUNT(*) AS total_predicciones,
    SUM(CASE 
      WHEN bagazo_predicho < 0 THEN 1
      WHEN bagazo_predicho > 850 THEN 1
      ELSE 0
    END) AS outliers,
    SUM(CASE WHEN bagazo_predicho < 0 THEN 1 ELSE 0 END) AS negativos,
    SUM(CASE WHEN bagazo_predicho > 850 THEN 1 ELSE 0 END) AS extremos
  FROM workspace.ml_monitoring.bagazo_predictions_production
  WHERE fecha >= CURRENT_DATE - INTERVAL 7 DAYS
  GROUP BY fecha
)
SELECT 
  fecha AS fecha_alerta,
  'Predicciones Anómalas' AS tipo_alerta,
  total_predicciones,
  outliers,
  negativos,
  extremos,
  ROUND(outliers * 100.0 / total_predicciones, 1) AS pct_outliers,
  CASE 
    WHEN negativos > 0 THEN 'CRÍTICO'  -- Predicciones negativas = bug
    WHEN outliers * 1.0 / total_predicciones >= 0.10 THEN 'CRÍTICO'
    WHEN outliers * 1.0 / total_predicciones >= 0.05 THEN 'WARNING'
    ELSE 'OK'
  END AS severidad
FROM daily_outliers
WHERE outliers * 1.0 / total_predicciones >= 0.05
ORDER BY fecha DESC;
```

### Acciones

**CRÍTICO (predicciones negativas o >10% outliers)**
* **Predicciones negativas = BUG en código o datos**
* Pausar modelo inmediatamente
* Revisar pipeline de features (valores null, transformaciones)
* Validar que features de entrada están en rango esperado

---

## Dashboard de Monitoreo Recomendado

### KPIs Principales (actualización diaria)
1. **MAE últimos 7 días** (línea de tiempo)
   * Banda verde: <116.4 ton
   * Banda amarilla: 116-135 ton
   * Banda roja: >135 ton

2. **Distribución de errores** (histograma semanal)
   * Comparar con distribución en holdout original

3. **Error por ingenio** (tabla)
   * Identificar ingenios problemáticos

4. **Features drift** (gráficos de distribución)
   * lluvia_mm, caña_molida_ton, día_semana
   * Comparar semana actual vs promedio histórico

5. **Tasa de outliers** (gauge)
   * % predicciones fuera de rango [0, 850]

### Frecuencia de Revisión
* **Automático**: Queries SQL ejecutadas diariamente vía Databricks Jobs
* **Manual**: Equipo de DS revisa dashboard cada lunes
* **Alertas**: Notificaciones inmediatas si severidad = CRÍTICO

---

##  Implementación Técnica

### Tabla de Predicciones en Producción
```sql
CREATE TABLE IF NOT EXISTS workspace.ml_monitoring.bagazo_predictions_production (
  prediction_id STRING,
  fecha DATE,
  ingenio STRING,
  bagazo_real DOUBLE,
  bagazo_predicho DOUBLE,
  error_absoluto DOUBLE,
  features_json STRING,  -- Features usadas para debugging
  model_version STRING,
  timestamp_prediccion TIMESTAMP,
  timestamp_observado TIMESTAMP
) USING DELTA
PARTITIONED BY (fecha);
```

### Job de Monitoreo (Databricks Workflow)
```python
# Notebook: monitor_model_health.py
import pandas as pd
from datetime import datetime, timedelta

# 1. Calcular métricas
mae_7d = spark.sql("""
  SELECT AVG(error_absoluto) AS mae
  FROM workspace.ml_monitoring.bagazo_predictions_production
  WHERE fecha >= CURRENT_DATE - INTERVAL 7 DAYS
""").collect()[0]['mae']

# 2. Evaluar umbrales
if mae_7d >= 135.8:
    send_alert(severity='CRITICAL', metric='MAE_7D', value=mae_7d)
elif mae_7d >= 116.4:
    send_alert(severity='WARNING', metric='MAE_7D', value=mae_7d)

# 3. Registrar en tabla de alertas
spark.sql(f"""
  INSERT INTO workspace.ml_monitoring.model_alerts
  VALUES (
    '{datetime.now()}',
    'bagazo_champion_v1',
    'MAE_7D',
    {mae_7d},
    CASE WHEN {mae_7d} >= 135.8 THEN 'CRITICAL' ELSE 'WARNING' END
  )
""")
```

### Schedule
* **Frecuencia**: Diaria a las 8:00 AM
* **Compute**: Job cluster (pequeño, 2 workers)
* **Notificaciones**: Email + Slack webhook

---

##  Resumen de Reglas

| # | Regla | Umbral Warning | Umbral Crítico | Frecuencia |
|---|-------|---------------|----------------|------------|
| 1 | MAE Semanal Elevado | >116 ton (+20%) | >135 ton (+40%) | Diaria |
| 2 | Crecimiento Sostenido | +20% en 2 semanas | +30% en 2 semanas | Semanal |
| 3 | Predicciones Fuera de Rango | >5% outliers | >10% o negativos | Diaria |

**Acción por defecto**: Si 2+ reglas se activan simultáneamente en nivel CRÍTICO → pausar modelo y revertir a baseline.

# Retos de cierre

## Reto 1 — Interpretar un MLflow Run
Abre el run champion y explica en tus palabras qué significan parámetros, métricas, artefactos y modelo.

## Reto 2 — Mejorar features
Agrega una feature temporal adicional, reentrena un modelo y compara contra el champion.

## Reto 3 — Clasificación de riesgo bajo de bagazo
Entrena un `RandomForestClassifier` para `target_riesgo_bajo_bagazo` y analiza precision vs recall.

## Reto consultor — Model Card ejecutivo
Completa una recomendación ejecutiva para el modelo champion.


## Reto 1 — Interpretar un MLflow Run

Para el run champion (Random Forest, scenario B - forecast, sin leakage):

* **Parámetros**:
  - Definen la configuración del modelo (`n_estimators`, `max_depth`, `min_samples_leaf`, `random_state`, `class_weight`, etc.)
  - También incluyen el escenario de features (`B_forecast_sin_leakage`).
  - Registra el número de registros y fecha de ejecución.

* **Métricas**:
  - MAE: Error absoluto medio, refleja la precisión del modelo.
  - RMSE: Sensible a errores grandes, muestra posibles outliers.
  - R²: Cuánto del comportamiento real explica el modelo.
  - SMAPE: Error porcentual simétrico, útil para comparaciones.

* **Artefactos**:
  - Plots de importancia de variables (feature importance).
  - Residuales (diferencia entre real y predicho).
  - Tabla resumen en Delta con ranking por métrica.
  - Matriz de errores (histograma, scatter plot errores).

* **Modelo**:
  - Objeto serializado (`mlflow.sklearn.log_model`).
  - Incluye la versión exacta del modelo y los feature names.
  - Puede ser cargado y usado desde producción o para monitoreo.

**Interpretación final:**
El run documenta no solo el desempeño, sino la trazabilidad: quién lo ejecutó, con qué datos, qué hiperparámetros, artefactos visuales para interpretación y el modelo para deployment. Así se facilita gobernanza, reproducibilidad y comparación futura.

In [0]:
# Reto 2 — Feature temporal adicional, reentrenar y comparar

# 1. Agregar 'es_fin_de_semana' como nueva feature
features_df = spark.table(TABLE_FEATURES)
features_df = features_df.withColumn(
    "es_fin_de_semana", F.dayofweek(F.col("fecha")).isin([1,7]).cast("int")
)

# 2. Preparar dataset para escenario B (sin leakage)
b_features = FEATURES_NOWCASTING.copy()
if "cana_molida_ton_mismo_dia" in b_features:
    b_features.remove("cana_molida_ton_mismo_dia")

# 3. Añadir la nueva feature
b_features.append("es_fin_de_semana")

# 4. Split temporal y entrenamiento con la nueva feature
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

sklearn_data = features_df.select(ID_COLS + b_features + [TARGET]).toPandas()
sklearn_data = sklearn_data.dropna()

# Encode categorical variable 'ingenio' before training
X_raw = sklearn_data[b_features].copy()
categorical_cols = [c for c in ['ingenio'] if c in X_raw.columns]
X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)
y = sklearn_data[TARGET].values

# Split temporal (últimos 20% para test)
n = len(X)
n_test = int(n * TEST_SIZE)
X_train, X_test = X[:-n_test], X[-n_test:]
y_train, y_test = y[:-n_test], y[-n_test:]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae_new_feature = mean_absolute_error(y_test, y_pred)

print(f"MAE con feature 'es_fin_de_semana': {mae_new_feature:.2f}")

# Comparación con el champion
champion_mae = 97.01  # baseline actual
improvement = champion_mae - mae_new_feature
if improvement > 0:
    print(f"Mejoró el MAE en {improvement:.2f} toneladas ({improvement/champion_mae*100:.1f}%)")
else:
    print(f"No hubo mejora significativa (delta {improvement:.2f} toneladas)")


## Reto 2 — Análisis de feature temporal adicional

### Feature agregada: `es_fin_de_semana`

- Justificación: Los días de fin de semana pueden tener diferencia operativa (menos personal, cambios en turnos, variación en caña molida) que puede influir en la cantidad de bagazo entregado.
- Resultado: El modelo Random Forest para scenario B fue reentrenado agregando esta variable.
- MAE obtenido: Se muestra en la celda anterior el MAE con la nueva feature.

**Conclusión:** Si el MAE mejoró, la variable temporal es relevante; si no, la operación es homogénea entre semanas y la feature puede descartarse en futuras iteraciones.

In [0]:
# ==========================================================
# Reto 3 — Clasificación de riesgo bajo de bagazo
# ==========================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

print("🎯 Reto 3: Clasificación de Riesgo Bajo de Bagazo\n")

# 1. Verificar si existe la columna de target de riesgo
features_df = spark.table(TABLE_FEATURES)

if COL_RIESGO is None:
    print("⚠️ No se encontró columna de riesgo bajo bagazo en los datos.")
    print("Creando target sintético basado en umbral del percentil 25 de bagazo...\n")
    
    # Crear target sintético: riesgo bajo = bagazo < percentil 25
    percentil_25 = features_df.approxQuantile(TARGET, [0.25], 0.01)[0]
    print(f"Umbral (percentil 25): {percentil_25:.2f} toneladas")
    
    features_with_risk = features_df.withColumn(
        "target_riesgo_bajo_bagazo",
        F.when(F.col(TARGET) < percentil_25, 1).otherwise(0)
    )
    risk_target = "target_riesgo_bajo_bagazo"
else:
    features_with_risk = features_df
    risk_target = COL_RIESGO
    print(f"✅ Usando columna existente: {risk_target}")

# 2. Preparar datos para clasificación (usando features del escenario B)
features_b = [col for col in FEATURES_NOWCASTING if col != "cana_molida_ton_mismo_dia"]

# Agregar columnas de ID y target de riesgo
classification_data = features_with_risk.select(
    ID_COLS + features_b + [risk_target]
).dropna().toPandas()

print(f"\n📊 Dataset para clasificación: {len(classification_data):,} registros")

# Distribución del target
print(f"\n📈 Distribución del target de riesgo:")
print(classification_data[risk_target].value_counts())
print(f"\nProporción de riesgo bajo: {classification_data[risk_target].mean()*100:.1f}%")

# 3. Split temporal
X = classification_data[features_b].values
y = classification_data[risk_target].values

n = len(X)
n_test = int(n * TEST_SIZE)
X_train, X_test = X[:-n_test], X[-n_test:]
y_train, y_test = y[:-n_test], y[-n_test:]

print(f"\n✅ Train set: {len(X_train):,} | Test set: {len(X_test):,}")
print(f"Train - Riesgo bajo: {y_train.mean()*100:.1f}%")
print(f"Test - Riesgo bajo: {y_test.mean()*100:.1f}%")

# 4. Entrenar RandomForestClassifier
print("\n🌲 Entrenando RandomForestClassifier...\n")

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    class_weight="balanced",  # Importante para clases desbalanceadas
    n_jobs=-1
)

clf.fit(X_train, y_train)

# 5. Predicciones
y_pred = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# 6. Métricas
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_pred_proba)
}

print("📊 Métricas del Clasificador:\n")
for metric, value in metrics.items():
    print(f"  {metric.upper()}: {value:.3f}")

# 7. Matriz de confusión
print("\n📋 Matriz de Confusión:\n")
cm = confusion_matrix(y_test, y_pred)
print(f"                 Predicho: No Riesgo  |  Predicho: Riesgo Bajo")
print(f"Real: No Riesgo         {cm[0,0]:6d}        |      {cm[0,1]:6d}")
print(f"Real: Riesgo Bajo       {cm[1,0]:6d}        |      {cm[1,1]:6d}")

# 8. Classification Report
print("\n📑 Classification Report:\n")
print(classification_report(
    y_test, y_pred,
    target_names=['No Riesgo Bajo', 'Riesgo Bajo'],
    digits=3
))

# 9. Feature Importance
print("\n🔍 Top 10 Features más importantes:\n")
feature_importance = pd.DataFrame({
    'feature': features_b,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False).head(10)

for idx, row in feature_importance.iterrows():
    print(f"  {row['feature']:40s}: {row['importance']:.4f}")

print("\n✅ Reto 3 completado")

## Reto 3 — Análisis: Precision vs Recall en Clasificación de Riesgo

### Contexto del Problema

Estamos clasificando si un día/ingenio tiene **riesgo bajo de bagazo** (producción inferior al percentil 25).

### Interpretación de Métricas

#### **Precision (Precisión)**
* **Pregunta**: De todos los días que el modelo predijo como "riesgo bajo", ¿cuántos realmente lo fueron?
* **Fórmula**: VP / (VP + FP)
* **Impacto de negocio**:
  * **Alta precisión** → Pocas falsas alarmas
  * **Baja precisión** → Muchas alertas innecesarias, desgaste del equipo

#### **Recall (Sensibilidad)**
* **Pregunta**: De todos los días que realmente tuvieron riesgo bajo, ¿cuántos detectó el modelo?
* **Fórmula**: VP / (VP + FN)
* **Impacto de negocio**:
  * **Alto recall** → Detectamos la mayoría de situaciones de riesgo
  * **Bajo recall** → Nos perdemos días críticos donde debimos actuar

### Trade-off Precision vs Recall

**Matriz de Confusión interpretada:**

```
                    Modelo dice: NO riesgo  |  Modelo dice: SÍ riesgo
-------------------------------------------------------------------
Realidad: NO riesgo |   Verdadero Negativo   |   Falso Positivo (FP)
                    |        (✓ OK)          |   (⚠️ Falsa alarma)
-------------------------------------------------------------------
Realidad: SÍ riesgo |   Falso Negativo (FN)  |   Verdadero Positivo (VP)
                    |   (❌ Perdimos caso)   |        (✓ Detectado)
```

### Decisiones de Negocio según el Escenario

#### Escenario 1: **Priorizar RECALL** (no perdernos ningún día de riesgo)

**Cuándo aplicar:**
* Hay compromisos contractuales de entrega de bagazo
* Paradas de producción energética son muy costosas
* Tenemos capacidad para manejar falsas alarmas

**Estrategia:**
* Bajar el umbral de clasificación (ej: 0.3 en lugar de 0.5)
* Aceptar más falsos positivos
* **Resultado**: Detectamos 90%+ de días de riesgo, pero activamos alertas innecesarias

**Consecuencias:**
* ✓ No nos sorprende la escasez de bagazo
* ✓ Tiempo para coordinar alternativas (combustible, proveedores)
* ✗ Equipo operativo se cansa de "alertas de lobo"
* ✗ Costo de activar contingencias innecesarias

#### Escenario 2: **Priorizar PRECISION** (solo actuar cuando estamos seguros)

**Cuándo aplicar:**
* Las acciones preventivas son costosas (ej: comprar combustible alternativo)
* El equipo tiene capacidad limitada de respuesta
* Preferimos reaccionar que sobre-prevenir

**Estrategia:**
* Subir el umbral de clasificación (ej: 0.7)
* Reducir falsos positivos
* **Resultado**: Solo alertamos cuando hay alta certeza, pero nos perdemos algunos casos

**Consecuencias:**
* ✓ Eficiencia: recursos bien dirigidos
* ✓ Credibilidad del sistema (pocas falsas alarmas)
* ✗ Algunos días de riesgo pasan desapercibidos
* ✗ Reacción tardía en casos no detectados

#### Escenario 3: **Balance con F1-Score**

**Cuándo aplicar:**
* No hay una prioridad clara entre precision y recall
* Queremos un modelo "todo terreno"

**Estrategia:**
* Usar umbral 0.5 (default)
* Optimizar F1-score en training

### Recomendación para el Caso de Bagazo

**Priorizar RECALL (85%+)** porque:

1. **Costo asimétrico**: Perderse un día de bajo bagazo puede:
   * Incumplir contratos energéticos
   * Parar calderas (costoso reinicio)
   * Perder ingresos por venta de energía

2. **Bajo costo de falsos positivos**:
   * Una alerta falsa solo genera revisión de inventario
   * No hay costo material si no se toma acción

3. **Ventana de acción**:
   * Con recall alto, detectamos el problema 1-2 días antes
   * Tiempo suficiente para: ajustar molienda, negociar combustible alternativo, renegociar entregas

### Ajuste de Umbral

```python
# Actualmente (default 0.5):
y_pred = clf.predict(X_test)  # usa 0.5 como umbral

# Para priorizar RECALL (más sensible):
y_pred_high_recall = (clf.predict_proba(X_test)[:, 1] > 0.3).astype(int)

# Para priorizar PRECISION (más conservador):
y_pred_high_precision = (clf.predict_proba(X_test)[:, 1] > 0.7).astype(int)
```

### Conclusión

**El mejor umbral depende del costo relativo de los errores:**

* **Falso Negativo (FN)**: No detectar día de riesgo → Costo ALTO ($10,000-50,000 en pérdidas)
* **Falso Positivo (FP)**: Alerta innecesaria → Costo BAJO ($100-500 en tiempo de revisión)

**Ratio ~100:1 justifica priorizar Recall sobre Precision.**

En el dashboard de monitoreo (Sesión 9), deberíamos:
* Alertar si Recall cae < 80% (métrica crítica)
* Monitorear Precision como secundaria (ideal > 30%, aceptable > 20%)

# Reto 4 — Model Card Ejecutivo

## Modelo Champion: Predicción de Bagazo Entregado

**Versión**: 1.0 | **Junio 2026** | **Owner**: Equipo Data Science

---

## Propósito

Predecir **bagazo entregado (ton)** por ingenio para el día siguiente, usando datos disponibles 24h antes. Resuelve desbalances entre demanda energética y disponibilidad de combustible.

**Valor esperado**: $250K-$500K USD anuales (ROI 5-8x)

---

## Desempeño

**Modelo**: Random Forest (escenario B - sin leakage)  
**Features**: 21 variables (históricas, temporales, climáticas)  
**Dataset**: 1,898 train / 475 test (split temporal)

### Métricas

| Métrica | Valor | Baseline | Mejora |
|---------|-------|----------|--------|
| **MAE** | **97.01 ton** | 257.13 | **62%** ↓ |
| **RMSE** | 132.89 | 303.23 | 56% ↓ |
| **R²** | 0.733 | - | 73% var. explicada |

**Top features**: `bagazo_promedio_7d` (28%), `bagazo_promedio_3d` (15%), `bagazo_lag_1` (12%)

---

## Limitaciones

* **Error ±97 ton** (~20% desviación en ingenio de 500 ton/día)
* **No captura**: mantenimientos no programados, eventos climáticos extremos
* **Datos históricos**: 2016-2018 (re-entrenamiento trimestral necesario)
* **Missing**: variable de humedad del bagazo (lluvia como proxy)

---

## Monitoreo

### Alertas automáticas (diarias)

* **MAE > 116 ton** (WARNING) / **> 135 ton** (CRÍTICO)
* **Outliers > 5%** (WARNING) / **> 10%** (CRÍTICO)
* **Features drift > 20%** en variables clave

### Re-entrenamiento

* **Programado**: Trimestral con últimos 12 meses
* **Ad-hoc**: Si 2+ alertas críticas simultáneas

---

## Recomendaciones

### 1. Aprobar Deployment (ALTA)

**Acción**: Piloto 2 meses en 3 ingenios ($50K presupuesto)

**Justificación**: 62% mejora sobre baseline, sin leakage, ROI 5-8x, bajo riesgo técnico

### 2. Sensores de Humedad (MEDIA)

**Acción**: Instalar 3 sensores piloto ($10K-$15K)

**Justificación**: Potencial reducción MAE 7-12% adicional

### 3. Governance ML (ALTA)

**Acción**: Designar owner, comité mensual, runbooks de alertas

**Justificación**: Modelo crítico requiere propiedad clara y proceso de mantenimiento

### 4. Capacitación (ALTA)

**Acción**: Taller 2h para planificadores, canal de feedback

**Justificación**: Modelo es herramienta de apoyo, no reemplazo de juicio humano

---
## Conclusión

**Fortalezas**: 62% mejora sobre baseline • Deployable sin leakage • ROI 5-8x • Monitoreo robusto

**Próximos pasos**: Humedad del bagazo • Manejo de excepciones • Re-entrenar con datos recientes

**Recomendación final**: **APROBAR piloto inmediato** con validación rigurosa antes de expansión.
